# Care Coordination Knowledge Graph — Neo4j Loader

This notebook loads the parsed FHIR CSV data into Neo4j as a knowledge graph.

## Graph Structure (Option B — Rich Graph)

**Nodes:** Patient, Disease, Encounter, Provider, Organization, Claim, Payer

**Relationships:**
- Patient → HAS_CONDITION → Disease
- Patient → HAS_ENCOUNTER → Encounter
- Encounter → WITH_PROVIDER → Provider
- Encounter → FOR_REASON → Disease
- Provider → WORKS_AT → Organization
- Encounter → HAS_CLAIM → Claim
- Claim → PAID_BY → Payer
- Patient → SEEN_BY → Provider (derived shortcut)

## 1. Setup & Connection

In [49]:
import pandas as pd
from neo4j import GraphDatabase
import os

# ========== CONFIGURATION ==========
NEO4J_URI = "neo4j://127.0.0.1:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "12345678"  # <-- CHANGE THIS to your Neo4j password

# Path to CSV files
DATA_DIR = os.path.join("..", "output")

# ========== CONNECT ==========
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# Test connection
with driver.session() as session:
    result = session.run("RETURN 'Connected to Neo4j!' AS message")
    print(result.single()["message"])

Connected to Neo4j!


## 2. Load CSV Data

In [23]:
# Load all CSVs
patients_df = pd.read_csv(os.path.join(DATA_DIR, "patients.csv"))
diseases_df = pd.read_csv(os.path.join(DATA_DIR, "diseases.csv"))
encounters_df = pd.read_csv(os.path.join(DATA_DIR, "encounters.csv"))
providers_df = pd.read_csv(os.path.join(DATA_DIR, "providers.csv"))
claims_df = pd.read_csv(os.path.join(DATA_DIR, "claims.csv"))
payers_df = pd.read_csv(os.path.join(DATA_DIR, "payers.csv"))

print(f"Patients:   {len(patients_df)} rows")
print(f"Diseases:   {len(diseases_df)} rows")
print(f"Encounters: {len(encounters_df)} rows")
print(f"Providers:  {len(providers_df)} rows")
print(f"Claims:     {len(claims_df)} rows")
print(f"Payers:     {len(payers_df)} rows")
print()

# Unique disease names for Disease nodes
unique_diseases = diseases_df["disease"].dropna().unique()
print(f"Unique diseases: {len(unique_diseases)}")

# Unique organizations for Organization nodes
orgs_df = providers_df[["service_provider", "service_provider_id"]].drop_duplicates()
print(f"Unique organizations: {len(orgs_df)}")

# Unique payers
unique_payers = payers_df["payer"].dropna().unique()
print(f"Unique payers: {len(unique_payers)}")

Patients:   117 rows
Diseases:   5858 rows
Encounters: 8316 rows
Providers:  238 rows
Claims:     14176 rows
Payers:     28352 rows

Unique diseases: 120
Unique organizations: 238
Unique payers: 10


## 3. Clear Existing Data (Optional)

⚠️ **Run this only if you want to start fresh. It deletes ALL data in the database.**

In [51]:
def clear_database(driver):
    """Delete all nodes and relationships in the database."""
    with driver.session() as session:
        session.run("MATCH (n) DETACH DELETE n")
    print("Database cleared.")

# Uncomment the line below to clear the database
clear_database(driver)

Database cleared.


## 4. Create Constraints & Indexes

Constraints ensure uniqueness and create indexes automatically for fast lookups.

In [25]:
def create_constraints(driver):
    """Create uniqueness constraints and indexes."""
    constraints = [
        "CREATE CONSTRAINT patient_id IF NOT EXISTS FOR (p:Patient) REQUIRE p.patient_id IS UNIQUE",
        "CREATE CONSTRAINT disease_name IF NOT EXISTS FOR (d:Disease) REQUIRE d.name IS UNIQUE",
        "CREATE CONSTRAINT encounter_id IF NOT EXISTS FOR (e:Encounter) REQUIRE e.encounter_id IS UNIQUE",
        "CREATE CONSTRAINT provider_id IF NOT EXISTS FOR (pr:Provider) REQUIRE pr.provider_id IS UNIQUE",
        "CREATE CONSTRAINT org_id IF NOT EXISTS FOR (o:Organization) REQUIRE o.org_id IS UNIQUE",
        "CREATE CONSTRAINT claim_id IF NOT EXISTS FOR (c:Claim) REQUIRE c.claim_id IS UNIQUE",
        "CREATE CONSTRAINT payer_name IF NOT EXISTS FOR (pay:Payer) REQUIRE pay.name IS UNIQUE",
    ]
    with driver.session() as session:
        for c in constraints:
            try:
                session.run(c)
                print(f"✅ {c.split('FOR')[0].strip()}")
            except Exception as e:
                print(f"⚠️  Constraint may already exist: {e}")

create_constraints(driver)

✅ CREATE CONSTRAINT patient_id IF NOT EXISTS
✅ CREATE CONSTRAINT disease_name IF NOT EXISTS
✅ CREATE CONSTRAINT encounter_id IF NOT EXISTS
✅ CREATE CONSTRAINT provider_id IF NOT EXISTS
✅ CREATE CONSTRAINT org_id IF NOT EXISTS
✅ CREATE CONSTRAINT claim_id IF NOT EXISTS
✅ CREATE CONSTRAINT payer_name IF NOT EXISTS


## 5. Load Nodes

### 5a. Patient Nodes

In [26]:
def load_patients(driver, patients_df):
    """Create Patient nodes from patients.csv"""
    query = """
    UNWIND $patients AS patient
    MERGE (p:Patient {patient_id: patient.patient_id})
    SET p.name = patient.name,
        p.gender = patient.gender,
        p.birth_date = patient.birth_date,
        p.marital_status = patient.marital_status,
        p.city = patient.city,
        p.state = patient.state,
        p.deceased = patient.deceased
    """
    # Convert DataFrame to list of dicts, replacing NaN with None
    patients = patients_df.where(patients_df.notna(), None).to_dict("records")
    
    with driver.session() as session:
        session.run(query, patients=patients)
    
    print(f"✅ Loaded {len(patients)} Patient nodes")

load_patients(driver, patients_df)

✅ Loaded 117 Patient nodes


### 5b. Disease Nodes

In [27]:
def load_diseases(driver, unique_diseases):
    """Create Disease nodes from unique disease names."""
    query = """
    UNWIND $diseases AS disease_name
    MERGE (d:Disease {name: disease_name})
    """
    with driver.session() as session:
        session.run(query, diseases=list(unique_diseases))
    
    print(f"✅ Loaded {len(unique_diseases)} Disease nodes")

load_diseases(driver, unique_diseases)

✅ Loaded 120 Disease nodes


### 5c. Provider Nodes

In [28]:
def load_providers(driver, providers_df):
    """Create Provider nodes from providers.csv"""
    query = """
    UNWIND $providers AS provider
    MERGE (pr:Provider {provider_id: provider.provider_id})
    SET pr.name = provider.name,
        pr.type = provider.type
    """
    # Deduplicate by provider_id (keep first occurrence)
    providers_unique = providers_df.drop_duplicates(subset="provider_id")
    providers = providers_unique.where(providers_unique.notna(), None).to_dict("records")
    
    with driver.session() as session:
        session.run(query, providers=providers)
    
    print(f"✅ Loaded {len(providers)} Provider nodes")

load_providers(driver, providers_df)

✅ Loaded 238 Provider nodes


### 5d. Organization Nodes

In [29]:
def load_organizations(driver, providers_df):
    """Create Organization nodes from unique service providers."""
    query = """
    UNWIND $orgs AS org
    MERGE (o:Organization {org_id: org.service_provider_id})
    SET o.name = org.service_provider
    """
    orgs_unique = providers_df[["service_provider", "service_provider_id"]].drop_duplicates()
    orgs = orgs_unique.where(orgs_unique.notna(), None).to_dict("records")
    
    with driver.session() as session:
        session.run(query, orgs=orgs)
    
    print(f"✅ Loaded {len(orgs)} Organization nodes")

load_organizations(driver, providers_df)

✅ Loaded 238 Organization nodes


### 5e. Encounter Nodes

In [30]:
def load_encounters(driver, encounters_df):
    """Create Encounter nodes from encounters.csv"""
    query = """
    UNWIND $encounters AS enc
    MERGE (e:Encounter {encounter_id: enc.encounter_id})
    SET e.start_date = enc.period_start,
        e.end_date = enc.period_end,
        e.reason = enc.reason
    """
    # Process in batches of 1000 to avoid memory issues
    batch_size = 1000
    total = len(encounters_df)
    
    for i in range(0, total, batch_size):
        batch = encounters_df.iloc[i:i+batch_size]
        encounters = batch.where(batch.notna(), None).to_dict("records")
        with driver.session() as session:
            session.run(query, encounters=encounters)
        print(f"  Batch {i//batch_size + 1}: loaded {min(i+batch_size, total)}/{total}", end="\r")
    
    print(f"\n✅ Loaded {total} Encounter nodes")

load_encounters(driver, encounters_df)

  Batch 9: loaded 8316/8316
✅ Loaded 8316 Encounter nodes


### 5f. Claim Nodes

In [31]:
def load_claims(driver, claims_df):
    """Create Claim nodes from claims.csv"""
    query = """
    UNWIND $claims AS claim
    MERGE (c:Claim {claim_id: claim.claim_id})
    SET c.amount = claim.amount,
        c.billing_start = claim.billable_period_start,
        c.billing_end = claim.billable_period_end
    """
    batch_size = 1000
    total = len(claims_df)
    
    for i in range(0, total, batch_size):
        batch = claims_df.iloc[i:i+batch_size]
        claims = batch.where(batch.notna(), None).to_dict("records")
        with driver.session() as session:
            session.run(query, claims=claims)
        print(f"  Batch {i//batch_size + 1}: {min(i+batch_size, total)}/{total}", end="\r")
    
    print(f"\n✅ Loaded {total} Claim nodes")

load_claims(driver, claims_df)

  Batch 15: 14176/14176
✅ Loaded 14176 Claim nodes


### 5g. Payer Nodes

In [32]:
def load_payers(driver, unique_payers):
    """Create Payer nodes from unique payer names."""
    query = """
    UNWIND $payers AS payer_name
    MERGE (pay:Payer {name: payer_name})
    """
    with driver.session() as session:
        session.run(query, payers=list(unique_payers))
    
    print(f"✅ Loaded {len(unique_payers)} Payer nodes")

load_payers(driver, unique_payers)

✅ Loaded 10 Payer nodes


## 6. Create Relationships

### 6a. Patient → HAS_CONDITION → Disease

In [33]:
def create_has_condition(driver, diseases_df):
    """Create HAS_CONDITION relationships between patients and diseases."""
    query = """
    UNWIND $records AS rec
    MATCH (p:Patient {patient_id: rec.patient_id})
    MATCH (d:Disease {name: rec.disease})
    MERGE (p)-[r:HAS_CONDITION]->(d)
    ON CREATE SET r.encounter_id = rec.encounter_id
    """
    # Deduplicate: one HAS_CONDITION per patient-disease pair
    deduped = diseases_df.drop_duplicates(subset=["patient_id", "disease"])
    
    batch_size = 500
    total = len(deduped)
    
    for i in range(0, total, batch_size):
        batch = deduped.iloc[i:i+batch_size]
        records = batch.where(batch.notna(), None).to_dict("records")
        with driver.session() as session:
            session.run(query, records=records)
        print(f"  Batch {i//batch_size + 1}: {min(i+batch_size, total)}/{total}", end="\r")
    
    print(f"\n✅ Created {total} HAS_CONDITION relationships")

create_has_condition(driver, diseases_df)

  Batch 3: 1080/1080
✅ Created 1080 HAS_CONDITION relationships


### 6b. Patient → HAS_ENCOUNTER → Encounter

In [34]:
def create_has_encounter(driver, encounters_df):
    """Create HAS_ENCOUNTER relationships between patients and encounters."""
    query = """
    UNWIND $records AS rec
    MATCH (p:Patient {patient_id: rec.patient_id})
    MATCH (e:Encounter {encounter_id: rec.encounter_id})
    MERGE (p)-[:HAS_ENCOUNTER]->(e)
    """
    batch_size = 1000
    total = len(encounters_df)
    
    for i in range(0, total, batch_size):
        batch = encounters_df.iloc[i:i+batch_size]
        records = batch[["patient_id", "encounter_id"]].to_dict("records")
        with driver.session() as session:
            session.run(query, records=records)
        print(f"  Batch {i//batch_size + 1}: {min(i+batch_size, total)}/{total}", end="\r")
    
    print(f"\n✅ Created {total} HAS_ENCOUNTER relationships")

create_has_encounter(driver, encounters_df)

  Batch 9: 8316/8316
✅ Created 8316 HAS_ENCOUNTER relationships


### 6c. Encounter → WITH_PROVIDER → Provider

In [35]:
def create_with_provider(driver, encounters_df):
    """Create WITH_PROVIDER relationships between encounters and providers."""
    query = """
    UNWIND $records AS rec
    MATCH (e:Encounter {encounter_id: rec.encounter_id})
    MATCH (pr:Provider {provider_id: rec.provider_id})
    MERGE (e)-[:WITH_PROVIDER]->(pr)
    """
    # Filter out rows with null provider_id
    valid = encounters_df.dropna(subset=["provider_id"])
    
    batch_size = 1000
    total = len(valid)
    
    for i in range(0, total, batch_size):
        batch = valid.iloc[i:i+batch_size]
        records = batch[["encounter_id", "provider_id"]].to_dict("records")
        # Keep provider_id as int
        for r in records:
            r["provider_id"] = int(r["provider_id"])
        with driver.session() as session:
            session.run(query, records=records)
        print(f"  Batch {i//batch_size + 1}: {min(i+batch_size, total)}/{total}", end="\r")
    
    print(f"\n✅ Created {total} WITH_PROVIDER relationships")

create_with_provider(driver, encounters_df)

  Batch 9: 8316/8316
✅ Created 8316 WITH_PROVIDER relationships


### 6d. Encounter → FOR_REASON → Disease

In [36]:
def create_for_reason(driver, diseases_df):
    """Create FOR_REASON relationships between encounters and diseases."""
    query = """
    UNWIND $records AS rec
    MATCH (e:Encounter {encounter_id: rec.encounter_id})
    MATCH (d:Disease {name: rec.disease})
    MERGE (e)-[:FOR_REASON]->(d)
    """
    valid = diseases_df.dropna(subset=["disease"])
    
    batch_size = 500
    total = len(valid)
    
    for i in range(0, total, batch_size):
        batch = valid.iloc[i:i+batch_size]
        records = batch[["encounter_id", "disease"]].to_dict("records")
        with driver.session() as session:
            session.run(query, records=records)
        print(f"  Batch {i//batch_size + 1}: {min(i+batch_size, total)}/{total}", end="\r")
    
    print(f"\n✅ Created {total} FOR_REASON relationships")

create_for_reason(driver, diseases_df)

  Batch 12: 5858/5858
✅ Created 5858 FOR_REASON relationships


### 6e. Provider → WORKS_AT → Organization

In [37]:
def create_works_at(driver, providers_df):
    """Create WORKS_AT relationships between providers and organizations."""
    query = """
    UNWIND $records AS rec
    MATCH (pr:Provider {provider_id: rec.provider_id})
    MATCH (o:Organization {org_id: rec.service_provider_id})
    MERGE (pr)-[:WORKS_AT]->(o)
    """
    # Each provider-org pair (deduplicated)
    prov_org = providers_df[["provider_id", "service_provider_id"]].drop_duplicates()
    prov_org = prov_org.dropna()
    records = prov_org.to_dict("records")
    
    # Keep provider_id as int
    for r in records:
        r["provider_id"] = int(r["provider_id"])
    
    with driver.session() as session:
        session.run(query, records=records)
    
    print(f"✅ Created {len(records)} WORKS_AT relationships")

create_works_at(driver, providers_df)

✅ Created 238 WORKS_AT relationships


### 6f. Encounter → HAS_CLAIM → Claim

In [38]:
def create_has_claim(driver, claims_df, encounters_df):
    """Link encounters to claims by matching patient_id and billing period to encounter period."""
    query = """
    UNWIND $records AS rec
    MATCH (e:Encounter {encounter_id: rec.encounter_id})
    MATCH (c:Claim {claim_id: rec.claim_id})
    MERGE (e)-[:HAS_CLAIM]->(c)
    """
    # Join claims with encounters on patient_id and matching time
    merged = pd.merge(
        encounters_df[["encounter_id", "patient_id", "period_start"]],
        claims_df[["claim_id", "patient_id", "billable_period_start"]],
        left_on=["patient_id", "period_start"],
        right_on=["patient_id", "billable_period_start"],
        how="inner"
    )
    
    batch_size = 1000
    total = len(merged)
    
    for i in range(0, total, batch_size):
        batch = merged.iloc[i:i+batch_size]
        records = batch[["encounter_id", "claim_id"]].to_dict("records")
        with driver.session() as session:
            session.run(query, records=records)
        print(f"  Batch {i//batch_size + 1}: {min(i+batch_size, total)}/{total}", end="\r")
    
    print(f"\n✅ Created {total} HAS_CLAIM relationships")

create_has_claim(driver, claims_df, encounters_df)

  Batch 15: 14311/14311
✅ Created 14311 HAS_CLAIM relationships


### 6g. Claim → PAID_BY → Payer

In [39]:
def create_paid_by(driver, payers_df):
    """Link claims to payers."""
    query = """
    UNWIND $records AS rec
    MATCH (c:Claim {claim_id: rec.claim_id})
    MATCH (pay:Payer {name: rec.payer})
    MERGE (c)-[:PAID_BY]->(pay)
    """
    batch_size = 1000
    total = len(payers_df)
    
    for i in range(0, total, batch_size):
        batch = payers_df.iloc[i:i+batch_size]
        records = batch.to_dict("records")
        with driver.session() as session:
            session.run(query, records=records)
        print(f"  Batch {i//batch_size + 1}: {min(i+batch_size, total)}/{total}", end="\r")
    
    print(f"\n✅ Created {total} PAID_BY relationships")

create_paid_by(driver, payers_df)

  Batch 29: 28352/28352
✅ Created 28352 PAID_BY relationships


### 6h. Patient → SEEN_BY → Provider (Derived Shortcut)

This is a convenience relationship computed from the encounter path.
It stores aggregated stats like visit count and last visit date.

In [40]:
def create_seen_by(driver):
    """Create derived SEEN_BY relationships with aggregated stats."""
    query = """
    MATCH (p:Patient)-[:HAS_ENCOUNTER]->(e:Encounter)-[:WITH_PROVIDER]->(pr:Provider)
    WITH p, pr, COUNT(e) AS visit_count, MAX(e.start_date) AS last_visit
    MERGE (p)-[r:SEEN_BY]->(pr)
    SET r.visit_count = visit_count,
        r.last_visit = last_visit
    """
    with driver.session() as session:
        result = session.run(query)
        summary = result.consume()
    
    # Count how many SEEN_BY edges were created
    with driver.session() as session:
        count = session.run("MATCH ()-[r:SEEN_BY]->() RETURN COUNT(r) AS cnt").single()["cnt"]
    
    print(f"✅ Created {count} SEEN_BY relationships (derived)")

create_seen_by(driver)

✅ Created 404 SEEN_BY relationships (derived)


## 7. Verify the Graph

In [41]:
def verify_graph(driver):
    """Print graph statistics."""
    node_queries = {
        "Patient nodes": "MATCH (p:Patient) RETURN COUNT(p) AS cnt",
        "Disease nodes": "MATCH (d:Disease) RETURN COUNT(d) AS cnt",
        "Encounter nodes": "MATCH (e:Encounter) RETURN COUNT(e) AS cnt",
        "Provider nodes": "MATCH (pr:Provider) RETURN COUNT(pr) AS cnt",
        "Organization nodes": "MATCH (o:Organization) RETURN COUNT(o) AS cnt",
        "Claim nodes": "MATCH (c:Claim) RETURN COUNT(c) AS cnt",
        "Payer nodes": "MATCH (pay:Payer) RETURN COUNT(pay) AS cnt",
    }
    
    rel_queries = {
        "HAS_CONDITION rels": "MATCH ()-[r:HAS_CONDITION]->() RETURN COUNT(r) AS cnt",
        "HAS_ENCOUNTER rels": "MATCH ()-[r:HAS_ENCOUNTER]->() RETURN COUNT(r) AS cnt",
        "WITH_PROVIDER rels": "MATCH ()-[r:WITH_PROVIDER]->() RETURN COUNT(r) AS cnt",
        "FOR_REASON rels": "MATCH ()-[r:FOR_REASON]->() RETURN COUNT(r) AS cnt",
        "WORKS_AT rels": "MATCH ()-[r:WORKS_AT]->() RETURN COUNT(r) AS cnt",
        "HAS_CLAIM rels": "MATCH ()-[r:HAS_CLAIM]->() RETURN COUNT(r) AS cnt",
        "PAID_BY rels": "MATCH ()-[r:PAID_BY]->() RETURN COUNT(r) AS cnt",
        "SEEN_BY rels": "MATCH ()-[r:SEEN_BY]->() RETURN COUNT(r) AS cnt",
    }
    
    print("=" * 45)
    print("  KNOWLEDGE GRAPH STATISTICS")
    print("=" * 45)
    
    with driver.session() as session:
        print("\n📦 NODES")
        print("-" * 35)
        for label, query in node_queries.items():
            count = session.run(query).single()["cnt"]
            print(f"  {label:<25} {count:>6}")
        
        print("\n🔗 RELATIONSHIPS")
        print("-" * 35)
        for label, query in rel_queries.items():
            count = session.run(query).single()["cnt"]
            print(f"  {label:<25} {count:>6}")
    
    print("\n" + "=" * 45)

verify_graph(driver)

  KNOWLEDGE GRAPH STATISTICS

📦 NODES
-----------------------------------
  Patient nodes                117
  Disease nodes                120
  Encounter nodes             8316
  Provider nodes               238
  Organization nodes           238
  Claim nodes                14176
  Payer nodes                   10

🔗 RELATIONSHIPS
-----------------------------------
  HAS_CONDITION rels          1080
  HAS_ENCOUNTER rels          8316
  WITH_PROVIDER rels          8316
  FOR_REASON rels             5858
  WORKS_AT rels                238
  HAS_CLAIM rels             14311
  PAID_BY rels               14176
  SEEN_BY rels                 404



## 8. Sample Queries — Test the Graph

### Q1: All providers for a specific patient

In [42]:
def query_patient_providers(driver, patient_name):
    """Find all providers who treated a patient."""
    query = """
    MATCH (p:Patient)-[r:SEEN_BY]->(pr:Provider)-[:WORKS_AT]->(o:Organization)
    WHERE p.name CONTAINS $name
    RETURN p.name AS patient,
           pr.name AS provider,
           o.name AS organization,
           r.visit_count AS visits,
           r.last_visit AS last_visit
    ORDER BY r.visit_count DESC
    """
    with driver.session() as session:
        result = session.run(query, name=patient_name)
        records = [dict(r) for r in result]
    
    if records:
        df = pd.DataFrame(records)
        print(f"\nProviders for patient matching '{patient_name}':")
        display(df)
    else:
        print(f"No results for '{patient_name}'")
    return records

# Test with first patient name
first_patient = patients_df.iloc[0]["name"].split()[0]
query_patient_providers(driver, first_patient)


Providers for patient matching 'Soledad678':


,patient,provider,organization,visits,last_visit
0,Soledad678 White193,Dr. Domitila545 Vandervort697,BETH ISRAEL DEACONESS HOSPITAL MILTON INC,43,2020-07-25T04:04:03+00:00
1,Soledad678 White193,Dr. Charles364 Prosacco716,CARING HEALTHCARE SERVICES LLC,20,2020-08-17T01:54:10+00:00
2,Soledad678 White193,Dr. Refugio197 Crooks415,MATTAPAN COMMUNITY HEALTH CENTER,2,2019-03-22T03:19:11+00:00


[{'patient': 'Soledad678 White193',
  'provider': 'Dr. Domitila545 Vandervort697',
  'organization': 'BETH ISRAEL DEACONESS HOSPITAL MILTON INC',
  'visits': 43,
  'last_visit': '2020-07-25T04:04:03+00:00'},
 {'patient': 'Soledad678 White193',
  'provider': 'Dr. Charles364 Prosacco716',
  'organization': 'CARING HEALTHCARE SERVICES LLC',
  'visits': 20,
  'last_visit': '2020-08-17T01:54:10+00:00'},
 {'patient': 'Soledad678 White193',
  'provider': 'Dr. Refugio197 Crooks415',
  'organization': 'MATTAPAN COMMUNITY HEALTH CENTER',
  'visits': 2,
  'last_visit': '2019-03-22T03:19:11+00:00'}]

### Q2: Top 10 most common diseases

In [43]:
def query_top_diseases(driver, top_n=10):
    """Find the most common diseases by patient count."""
    query = """
    MATCH (p:Patient)-[:HAS_CONDITION]->(d:Disease)
    RETURN d.name AS disease, COUNT(DISTINCT p) AS patient_count
    ORDER BY patient_count DESC
    LIMIT $top_n
    """
    with driver.session() as session:
        result = session.run(query, top_n=top_n)
        records = [dict(r) for r in result]
    
    df = pd.DataFrame(records)
    print(f"\nTop {top_n} diseases by patient count:")
    display(df)
    return records

query_top_diseases(driver)


Top 10 diseases by patient count:


,disease,patient_count
0,Patient referral for dental care (procedure),103
1,Gingivitis (disorder),98
2,Viral sinusitis (disorder),71
3,Acute viral pharyngitis (disorder),48
4,Acute bronchitis (disorder),48
5,Screening for malignant neoplasm of colon (pro...,41
6,Chronic pain (finding),36
7,Contraception care (regime/therapy),36
8,Normal pregnancy (finding),25
9,Anemia (disorder),24


[{'disease': 'Patient referral for dental care (procedure)',
  'patient_count': 103},
 {'disease': 'Gingivitis (disorder)', 'patient_count': 98},
 {'disease': 'Viral sinusitis (disorder)', 'patient_count': 71},
 {'disease': 'Acute viral pharyngitis (disorder)', 'patient_count': 48},
 {'disease': 'Acute bronchitis (disorder)', 'patient_count': 48},
 {'disease': 'Screening for malignant neoplasm of colon (procedure)',
  'patient_count': 41},
 {'disease': 'Chronic pain (finding)', 'patient_count': 36},
 {'disease': 'Contraception care (regime/therapy)', 'patient_count': 36},
 {'disease': 'Normal pregnancy (finding)', 'patient_count': 25},
 {'disease': 'Anemia (disorder)', 'patient_count': 24}]

### Q3: Patients with most conditions (potential high-risk)

In [44]:
def query_high_condition_patients(driver, top_n=10):
    """Find patients with the most conditions."""
    query = """
    MATCH (p:Patient)-[:HAS_CONDITION]->(d:Disease)
    WITH p, COUNT(d) AS condition_count, COLLECT(d.name) AS conditions
    RETURN p.name AS patient,
           p.gender AS gender,
           p.birth_date AS birth_date,
           condition_count,
           conditions[0..5] AS top_conditions
    ORDER BY condition_count DESC
    LIMIT $top_n
    """
    with driver.session() as session:
        result = session.run(query, top_n=top_n)
        records = [dict(r) for r in result]
    
    df = pd.DataFrame(records)
    print(f"\nTop {top_n} patients by condition count:")
    display(df)
    return records

query_high_condition_patients(driver)


Top 10 patients by condition count:


,patient,gender,birth_date,condition_count,top_conditions
0,Emerald468 Botsford977,female,1948-10-08,22,"[Allergic disposition (finding), Loss of teeth..."
1,Werner409 Cruickshank494,male,1960-11-26,21,"[Loss of teeth (disorder), Patient referral fo..."
2,Andrew29 Williamson769,male,1946-07-01,18,"[Allergic disposition (finding), Loss of teeth..."
3,Soledad678 White193,female,1950-05-01,18,"[Cow's milk (substance), Allergic disposition ..."
4,Chantelle310 Oberbrunner298,female,1974-12-17,18,"[Patient referral for dental care (procedure),..."
5,Giovanni385 Paucek755,male,1960-11-26,17,"[Patient referral for dental care (procedure),..."
6,Damian46 Bauch723,male,1964-01-03,17,"[Allergic disposition (finding), Asthma (disor..."
7,Rudy520 Padberg411,male,1968-04-27,15,"[Sleep disorder (disorder), Patient referral f..."
8,Lazaro919 Lang846,male,1960-10-19,15,"[Loss of teeth (disorder), Patient referral fo..."
9,Jerrie417 Marvin195,female,1977-11-06,15,"[Patient referral for dental care (procedure),..."


[{'patient': 'Emerald468 Botsford977',
  'gender': 'female',
  'birth_date': '1948-10-08',
  'condition_count': 22,
  'top_conditions': ['Allergic disposition (finding)',
   'Loss of teeth (disorder)',
   'Patient referral for dental care (procedure)',
   'Gingivitis (disorder)',
   'Concussion injury of brain (disorder)']},
 {'patient': 'Werner409 Cruickshank494',
  'gender': 'male',
  'birth_date': '1960-11-26',
  'condition_count': 21,
  'top_conditions': ['Loss of teeth (disorder)',
   'Patient referral for dental care (procedure)',
   'Gingivitis (disorder)',
   'Chronic pain (finding)',
   'Screening for malignant neoplasm of colon (procedure)']},
 {'patient': 'Andrew29 Williamson769',
  'gender': 'male',
  'birth_date': '1946-07-01',
  'condition_count': 18,
  'top_conditions': ['Allergic disposition (finding)',
   'Loss of teeth (disorder)',
   'Patient referral for dental care (procedure)',
   'Gingivitis (disorder)',
   'Chronic pain (finding)']},
 {'patient': 'Soledad678 Whi

### Q4: Shared patients between providers

In [45]:
def query_provider_overlap(driver, top_n=10):
    """Find provider pairs with the most shared patients."""
    query = """
    MATCH (pr1:Provider)<-[:SEEN_BY]-(p:Patient)-[:SEEN_BY]->(pr2:Provider)
    WHERE elementId(pr1) < elementId(pr2)
    WITH pr1, pr2, COUNT(DISTINCT p) AS shared_patients
    RETURN pr1.name AS provider_1,
           pr2.name AS provider_2,
           shared_patients
    ORDER BY shared_patients DESC
    LIMIT $top_n
    """
    with driver.session() as session:
        result = session.run(query, top_n=top_n)
        records = [dict(r) for r in result]
    
    df = pd.DataFrame(records)
    print(f"\nTop {top_n} provider pairs by shared patients:")
    display(df)
    return records

query_provider_overlap(driver)


Top 10 provider pairs by shared patients:


,provider_1,provider_2,shared_patients
0,Dr. Lynwood354 Tromp100,Dr. Donnell534 D'Amore443,4
1,Dr. Gabriel934 Reilly981,Dr. Courtney281 Kihn564,3
2,Dr. Chris95 Kub800,Dr. Winston220 McCullough561,3
3,Dr. Gabriel934 Reilly981,Dr. Kendra609 Kassulke119,3
4,Dr. Gertrud593 Kuhic920,Dr. Theo630 Rohan584,3
5,Dr. Quentin28 Fritsch593,Dr. Sean831 DuBuque211,3
6,Dr. Lynwood354 Tromp100,Dr. Wanda961 Goyette777,3
7,Dr. Donnell534 D'Amore443,Dr. Wanda961 Goyette777,3
8,Dr. Domenic627 Blick895,Dr. Cassi819 McClure239,3
9,Dr. Quentin28 Fritsch593,Dr. Miyoko154 Paucek755,3


[{'provider_1': 'Dr. Lynwood354 Tromp100',
  'provider_2': "Dr. Donnell534 D'Amore443",
  'shared_patients': 4},
 {'provider_1': 'Dr. Gabriel934 Reilly981',
  'provider_2': 'Dr. Courtney281 Kihn564',
  'shared_patients': 3},
 {'provider_1': 'Dr. Chris95 Kub800',
  'provider_2': 'Dr. Winston220 McCullough561',
  'shared_patients': 3},
 {'provider_1': 'Dr. Gabriel934 Reilly981',
  'provider_2': 'Dr. Kendra609 Kassulke119',
  'shared_patients': 3},
 {'provider_1': 'Dr. Gertrud593 Kuhic920',
  'provider_2': 'Dr. Theo630 Rohan584',
  'shared_patients': 3},
 {'provider_1': 'Dr. Quentin28 Fritsch593',
  'provider_2': 'Dr. Sean831 DuBuque211',
  'shared_patients': 3},
 {'provider_1': 'Dr. Lynwood354 Tromp100',
  'provider_2': 'Dr. Wanda961 Goyette777',
  'shared_patients': 3},
 {'provider_1': "Dr. Donnell534 D'Amore443",
  'provider_2': 'Dr. Wanda961 Goyette777',
  'shared_patients': 3},
 {'provider_1': 'Dr. Domenic627 Blick895',
  'provider_2': 'Dr. Cassi819 McClure239',
  'shared_patients': 

### Q5: Disease co-occurrence (comorbidities)

In [46]:
def query_comorbidities(driver, top_n=10):
    """Find disease pairs that co-occur most frequently."""
    query = """
    MATCH (p:Patient)-[:HAS_CONDITION]->(d1:Disease),
          (p)-[:HAS_CONDITION]->(d2:Disease)
    WHERE elementId(d1) < elementId(d2)
    WITH d1.name AS disease_1, d2.name AS disease_2, COUNT(DISTINCT p) AS co_occurrence
    RETURN disease_1, disease_2, co_occurrence
    ORDER BY co_occurrence DESC
    LIMIT $top_n
    """
    with driver.session() as session:
        result = session.run(query, top_n=top_n)
        records = [dict(r) for r in result]
    
    df = pd.DataFrame(records)
    print(f"\nTop {top_n} disease co-occurrences:")
    display(df)
    return records

query_comorbidities(driver)


Top 10 disease co-occurrences:


,disease_1,disease_2,co_occurrence
0,Patient referral for dental care (procedure),Gingivitis (disorder),92
1,Patient referral for dental care (procedure),Viral sinusitis (disorder),62
2,Gingivitis (disorder),Viral sinusitis (disorder),60
3,Patient referral for dental care (procedure),Acute viral pharyngitis (disorder),43
4,Gingivitis (disorder),Acute viral pharyngitis (disorder),42
5,Gingivitis (disorder),Screening for malignant neoplasm of colon (pro...,41
6,Patient referral for dental care (procedure),Acute bronchitis (disorder),41
7,Patient referral for dental care (procedure),Screening for malignant neoplasm of colon (pro...,40
8,Gingivitis (disorder),Acute bronchitis (disorder),40
9,Gingivitis (disorder),Chronic pain (finding),35


[{'disease_1': 'Patient referral for dental care (procedure)',
  'disease_2': 'Gingivitis (disorder)',
  'co_occurrence': 92},
 {'disease_1': 'Patient referral for dental care (procedure)',
  'disease_2': 'Viral sinusitis (disorder)',
  'co_occurrence': 62},
 {'disease_1': 'Gingivitis (disorder)',
  'disease_2': 'Viral sinusitis (disorder)',
  'co_occurrence': 60},
 {'disease_1': 'Patient referral for dental care (procedure)',
  'disease_2': 'Acute viral pharyngitis (disorder)',
  'co_occurrence': 43},
 {'disease_1': 'Gingivitis (disorder)',
  'disease_2': 'Acute viral pharyngitis (disorder)',
  'co_occurrence': 42},
 {'disease_1': 'Gingivitis (disorder)',
  'disease_2': 'Screening for malignant neoplasm of colon (procedure)',
  'co_occurrence': 41},
 {'disease_1': 'Patient referral for dental care (procedure)',
  'disease_2': 'Acute bronchitis (disorder)',
  'co_occurrence': 41},
 {'disease_1': 'Patient referral for dental care (procedure)',
  'disease_2': 'Screening for malignant neo

### Q6: Total cost per patient (using Claims)

In [47]:
def query_patient_costs(driver, top_n=10):
    """Find patients with the highest total healthcare costs."""
    query = """
    MATCH (p:Patient)-[:HAS_ENCOUNTER]->(e:Encounter)-[:HAS_CLAIM]->(c:Claim)
    WITH p, SUM(c.amount) AS total_cost, COUNT(DISTINCT c) AS num_claims
    RETURN p.name AS patient,
           p.gender AS gender,
           round(total_cost, 2) AS total_cost,
           num_claims
    ORDER BY total_cost DESC
    LIMIT $top_n
    """
    with driver.session() as session:
        result = session.run(query, top_n=top_n)
        records = [dict(r) for r in result]
    
    df = pd.DataFrame(records)
    print(f"\nTop {top_n} patients by total cost:")
    display(df)
    return records

query_patient_costs(driver)


Top 10 patients by total cost:


,patient,gender,total_cost,num_claims
0,Giovanni385 Paucek755,male,3607841.24,791
1,Chad48 Gerhold939,male,2943910.30,633
2,Chantelle310 Oberbrunner298,female,2784332.83,625
3,Lazaro919 Lang846,male,790269.54,792
4,Petrina721 Douglas31,female,736159.33,101
5,Theda286 Herzog843,female,666322.26,1380
6,Emerald468 Botsford977,female,662775.64,1304
7,Broderick767 Abshire638,male,553062.22,659
8,Laveta191 Johns824,female,548751.59,79
9,Lekisha909 Bosco882,female,536099.85,101


[{'patient': 'Giovanni385 Paucek755',
  'gender': 'male',
  'total_cost': 3607841.24,
  'num_claims': 791},
 {'patient': 'Chad48 Gerhold939',
  'gender': 'male',
  'total_cost': 2943910.3,
  'num_claims': 633},
 {'patient': 'Chantelle310 Oberbrunner298',
  'gender': 'female',
  'total_cost': 2784332.83,
  'num_claims': 625},
 {'patient': 'Lazaro919 Lang846',
  'gender': 'male',
  'total_cost': 790269.54,
  'num_claims': 792},
 {'patient': 'Petrina721 Douglas31',
  'gender': 'female',
  'total_cost': 736159.33,
  'num_claims': 101},
 {'patient': 'Theda286 Herzog843',
  'gender': 'female',
  'total_cost': 666322.26,
  'num_claims': 1380},
 {'patient': 'Emerald468 Botsford977',
  'gender': 'female',
  'total_cost': 662775.64,
  'num_claims': 1304},
 {'patient': 'Broderick767 Abshire638',
  'gender': 'male',
  'total_cost': 553062.22,
  'num_claims': 659},
 {'patient': 'Laveta191 Johns824',
  'gender': 'female',
  'total_cost': 548751.59,
  'num_claims': 79},
 {'patient': 'Lekisha909 Bosco

### Q7: Cost breakdown by Payer

In [48]:
def query_payer_breakdown(driver):
    """Show total cost handled by each payer."""
    query = """
    MATCH (c:Claim)-[:PAID_BY]->(pay:Payer)
    WITH pay.name AS payer, SUM(c.amount) AS total_paid, COUNT(c) AS num_claims
    RETURN payer, round(total_paid, 2) AS total_paid, num_claims
    ORDER BY total_paid DESC
    """
    with driver.session() as session:
        result = session.run(query)
        records = [dict(r) for r in result]
    
    df = pd.DataFrame(records)
    print("\nPayer breakdown:")
    display(df)
    return records

query_payer_breakdown(driver)


Payer breakdown:


,payer,total_paid,num_claims
0,Medicare,9591625.63,4656
1,Medicaid,3862408.84,1735
2,Dual Eligible,3649263.99,1728
3,Humana,3267788.17,1137
4,Blue Cross Blue Shield,1860905.51,799
5,NO_INSURANCE,1615770.89,931
6,Cigna Health,1401975.16,709
7,UnitedHealthcare,1367959.40,1119
8,Anthem,1152566.24,795
9,Aetna,872495.11,567


[{'payer': 'Medicare', 'total_paid': 9591625.63, 'num_claims': 4656},
 {'payer': 'Medicaid', 'total_paid': 3862408.84, 'num_claims': 1735},
 {'payer': 'Dual Eligible', 'total_paid': 3649263.99, 'num_claims': 1728},
 {'payer': 'Humana', 'total_paid': 3267788.17, 'num_claims': 1137},
 {'payer': 'Blue Cross Blue Shield',
  'total_paid': 1860905.51,
  'num_claims': 799},
 {'payer': 'NO_INSURANCE', 'total_paid': 1615770.89, 'num_claims': 931},
 {'payer': 'Cigna Health', 'total_paid': 1401975.16, 'num_claims': 709},
 {'payer': 'UnitedHealthcare', 'total_paid': 1367959.4, 'num_claims': 1119},
 {'payer': 'Anthem', 'total_paid': 1152566.24, 'num_claims': 795},
 {'payer': 'Aetna', 'total_paid': 872495.11, 'num_claims': 567}]

## 9. Neo4j Browser Visualization Queries

Open **http://localhost:7474** and run these Cypher queries to visualize the graph:

```cypher
-- Full graph overview (limited to 100 nodes)
MATCH (n)-[r]->(m) RETURN n, r, m LIMIT 100

-- One patient's full network
MATCH (p:Patient)-[r]-(n) WHERE p.name CONTAINS 'Soledad' RETURN p, r, n

-- Disease network for a specific condition
MATCH (p:Patient)-[:HAS_CONDITION]->(d:Disease {name: 'Chronic kidney disease stage 4 (disorder)'})
MATCH (p)-[:SEEN_BY]->(pr:Provider)
RETURN p, d, pr

-- Provider-Organization network
MATCH (pr:Provider)-[:WORKS_AT]->(o:Organization) RETURN pr, o

-- Payer network — which payers cover which patients
MATCH (p:Patient)-[:HAS_ENCOUNTER]->(e:Encounter)-[:HAS_CLAIM]->(c:Claim)-[:PAID_BY]->(pay:Payer)
RETURN p, pay LIMIT 100

-- All payers and their claim counts
MATCH (c:Claim)-[:PAID_BY]->(pay:Payer) RETURN pay, COUNT(c) AS claims
```

## 10. Close Connection

In [ ]:
driver.close()
print("🔌 Neo4j connection closed.")